# End of week 1 exercise

Scrape, analyze, and summarize job descriptions from online websites to identify the most in-demand technical skills.

In [ ]:
# imports
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_contents

In [ ]:
# constants
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'


In [ ]:
# set up environment
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
else:
    print("API key founded")


In [ ]:
ai_engineer_url = [
    "https://careerviet.vn/vi/tim-viec-lam/senior-ai-engineer.35C6BE00.html"
]

words = fetch_website_contents(ai_engineer_url[0]).split()
print(len(words))

In [ ]:
print(fetch_website_contents(ai_engineer_url[0]).split())

In [ ]:
# here is the question; type over this to ask something new
system_prompt = """
Role: You are an expert Technical Recruiter and Job Analyst.

Task: Analyze the provided Job Description (JD) and extract all required skills (technical, soft skills, and tools). Focus on the required skill not company's benefits.

Requirements: > 1. Rank the skills in a numbered list from 1 (Most Critical) to N (Least Critical).
2. Ranking Logic: Base the rank on the frequency of mention, the "Required" vs. "Preferred" sections, and how central the skill is to the core responsibilities described.
3. Provide a brief (one-sentence) justification for why the top 3 skills were ranked highest.
"""


question = f"""
Here are the list of website's links about AI Engineering:
https://careerviet.vn/vi/tim-viec-lam/senior-ai-engineer.35C6BE00.html, https://careerviet.vn/vi/tim-viec-lam/senior-ai-engineer.35C68A6C.html,
https://careerviet.vn/vi/tim-viec-lam/ai-engineer-agentic-llm-systems.35C6CDE2.html, https://careerviet.vn/vi/tim-viec-lam/ai-engineer-computer-vision-npl-llm-khoi-cong-nghe-thong-tin-holt-13.35C6B8DD.html,
https://careerviet.vn/vi/tim-viec-lam/ai-engineer.35C6867E.html
"""

In [ ]:
def extract_urls(user_input):
    url_pattern = r'https?://[^\s,]+'
    urls = re.findall(url_pattern, user_input)
    
    return [url.rstrip('.,') for url in urls]

In [ ]:
extract_urls(question)

In [ ]:
def get_user_prompt(question):
    url_list = extract_urls(question)
    user_prompt = question + "Below are the job descrtion for each job: \n"
    for count, link in enumerate(url_list):
        user_prompt += f"Job Description {count+1}: " + fetch_website_contents(link) + "\n"
    return user_prompt

In [ ]:
user_prompt = get_user_prompt(question)

In [ ]:
print(user_prompt)

In [ ]:
# Get gpt-4o-mini to answer, with streaming
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
print(messages)

In [ ]:
# Get Llama 3.2 to answer
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

response = ollama.chat.completions.create(
    model = "llama3.2",
    messages = messages
)
print(response.choices[0].message.content)
